**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/06-bug-hunt.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/calibration-pitfalls) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6F · Bug Hunt 🐞 (pairs)

_Lecture 6, Common calibration pitfalls._

Eight short calibration programs, eight classic bugs, each one seen in real student (and professional) code.
For every bug: **predict** what will go wrong, **run** it, **fix** it, then run its check. The scoreboard counts to 8/8.

Work in pairs: one person drives, the other explains the fix before it is typed. Swap every bug.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)

import lab6_checks as C
C.fetch_set('gopro')
C.fetch_set('road_cal')
print("data ready")

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C

gopro = sorted(glob.glob("data/calibration_wide/GOPR*.jpg"))
road = sorted(glob.glob("data/camera_cal/*.jpg"))
img = cv2.imread(gopro[0]); gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

## 🐞 Bug 1: the board is never found

In [ ]:
# The GoPro board has 9 x 7 squares.
ret, corners = cv2.findChessboardCorners(gray, (9, 7), None)
print("found:", ret)

In [ ]:
C.check_bug1(ret)

## 🐞 Bug 2: a strange principal point

In [ ]:
objp = C.board_points(8, 6)
op, ip = [], []
for f in gopro[:12]:
    g = cv2.imread(f, 0); ok, c = C.find_corners(g, (8, 6))
    if ok: op.append(objp); ip.append(c)
ret, mtx, dist, _, _ = cv2.calibrateCamera(op, ip, g.shape, None, None)
print(mtx)

In [ ]:
C.check_bug2(mtx)

## 🐞 Bug 3: the RMS error is enormous

In [ ]:
nx, ny = 8, 6
objp = np.zeros((nx * ny, 3), np.float32)
objp[:, :2] = np.mgrid[0:ny, 0:nx].T.reshape(-1, 2)
op = [objp] * len(ip)
rms = cv2.calibrateCamera(op, ip, (1280, 960), None, None)[0]
print("RMS:", rms)

In [ ]:
C.check_bug3(rms)

## 🐞 Bug 4: an error about image sizes (road camera)

In [ ]:
objp = C.board_points(9, 6)
op, ip, size = [], [], None
for f in road:
    g = cv2.imread(f, 0)
    size = g.shape[::-1]
    ok, c = C.find_corners(g, (9, 6))
    if ok: op.append(objp); ip.append(c)
print(len(op), "views, last size", size)
try:
    rms = cv2.calibrateCamera(op, ip, size, None, None)[0]; print("RMS", rms)
except cv2.error as e:
    print("calibrateCamera failed:", str(e)[:120])

In [ ]:
C.check_bug4(len(op), size)

## 🐞 Bug 5: OpenCV 4 code on OpenCV 5

In [ ]:
ret, corners = cv2.findChessboardCorners(gray, (8, 6), None)
first_corner = corners[0][0]          # worked in OpenCV 4.x: (x, y)
print("first corner:", first_corner)

In [ ]:
C.check_bug5(first_corner)

## 🐞 Bug 6: a mirrored top-down view

In [ ]:
K, D = C.reference_gopro()
und = cv2.undistort(img, K, D, None, K)
g = cv2.cvtColor(und, cv2.COLOR_BGR2GRAY)
ok, c = C.find_corners(g, (8, 6)); c = c.reshape(-1, 2)
src = np.float32([c[0], c[7], c[-1], c[-8]])                         # TL, TR, BR, BL
dst = np.float32([[1180, 100], [100, 100], [100, 860], [1180, 860]])
M = cv2.getPerspectiveTransform(src, dst)
warped = cv2.warpPerspective(cv2.drawChessboardCorners(und.copy(), (8, 6), c, True), M, (1280, 960))
plt.figure(figsize=(8, 6)); plt.imshow(warped[:, :, ::-1]); plt.axis("off"); plt.show()
# the first corner (red) should be top-left: compare where it lands
p = cv2.perspectiveTransform(c[None, :1].astype(np.float64), M)[0, 0]
q = cv2.perspectiveTransform(c[None, 7:8].astype(np.float64), M)[0, 0]

In [ ]:
C.check_bug6(p[0], q[0])

## 🐞 Bug 7: undistorting a smaller image

In [ ]:
K, D = C.reference_gopro()
small = cv2.resize(img, None, fx=0.5, fy=0.5)          # 640 x 480 for speed
K_small = K.copy()
und_small = cv2.undistort(small, K_small, D, None, K_small)
plt.figure(figsize=(8, 6)); plt.imshow(und_small[:, :, ::-1]); plt.axis("off"); plt.show()

In [ ]:
C.check_bug7(K_small)

## 🐞 Bug 8: `cv2.fisheye.calibrate` refuses the points

In [ ]:
op, ip = [], []
for f in gopro[:15]:
    g = cv2.imread(f, 0); ok, c = C.find_corners(g, (8, 6))
    if ok: op.append(C.board_points(8, 6)); ip.append(c.reshape(-1, 2))
rms = None
try:
    rms, Kf, Df, _, _ = cv2.fisheye.calibrate(op, ip, (1280, 960), None, None,
                                              flags=C.fisheye_flags())
    print("fisheye RMS", rms)
except cv2.error as e:
    print("fisheye.calibrate failed:", str(e)[:160])

In [ ]:
C.check_bug8(rms)

## 🏁 Scoreboard

In [ ]:
C.bug_score()

## Reflection

For each bug write **one sentence**: what the symptom was, and the rule that prevents it. Which bugs give **no error**
at all, only wrong numbers? Those are the dangerous ones.